# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# 多層防御の底上げ —— 2026-09-15

**[10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) §7 の「利用者の判断待ち」14 項目の答えと、その当て方。**
手元のリポジトリにはすべて入れた。**2026-09-17 に本番へ配備した**(残りは §0 の「利用者待ちの作業」。# はその表の番号)。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

| §7 | 決まったこと | 手元 | 本番 | この文書 |
|---|---|---|---|---|
| 1 | Logto Console の MFA は A(SQL で当てる)。**MFA は既に Mandatory だった**ので、パスワード方針と総当たりロックを揃える | SQL を用意 | **済**(2026-09-18) | §4 |
| 2 | MariaDB の `root@%` を消し、`Main` を DB の網からだけに。**自分だけの最低権限アカウント**を phpMyAdmin の root で作る | 網の範囲を固定 | **済**(09-17。権限の範囲だけ残り #3) | §6 |
| 3 | Logto の Postgres 利用者を SUPERUSER でないロールへ | SQL と compose | **済**(2026-09-18。`logto_app`) | §3 |
| 4 | Soketi の Node 16 をやめる | Node 24 に載せ替え | **済**(2026-09-18。Node v24.21.0) | §2 |
| 5 | Test.zip を D:\ の控えへ | — | **済**(`D:\Backups\_archive`) | §7(ACL の後) |
| 6 | ランキングに溜まった利用者の行を消す | `--all-users` を足した | **済**(1 行) | §0 |
| 7 | `app/build (1).gradle.kts` と未使用の 4 ファイルを退避 | **済**(`Test/Old/unused-20260915`) | — | §0 |
| 8 | `allow-admin.conf` の `172.16.0.0/12` を外し、自分の IP へ | 済 | **済**(09-17) | §2 |
| 9 | 網の分割・read_only・cap_drop・src の :ro・digest 固定・管理画面の別オリジン・`__Host-` | 済 | **済**(09-17。別オリジンは 14 で) | §2・§5 |
| 10 | D:\Backups の ACL を本人と SYSTEM だけに | — | **済**(実測) | §7 |
| 11 | km が docker グループ経由で実質 root → 鍵にパスフレーズ、km の降格も視野に | — | **済**(2026-09-18。鍵にパスフレーズ + 門番 + km の降格まで) | §7 |
| 12 | KosenAPP の refresh_token の回転と絶対期限 | **済**(利用者) | — | — |
| 13 | reCAPTCHA の「ドメイン名の検証」 | **済**(利用者) | — | — |
| 14 | web のイメージから pdo_pgsql を外す | 済 | **済**(2026-09-18。web に pdo_pgsql は無い) | §2 |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 0. いまの状態

### 利用者待ちの作業(2026-09-18 02:30 に本番とこの PC を実測し直した)

**残っているのは #2(APK の配布)と #15(旧ドメインの解約)だけ。** ほかは済み。

**「任せる」は、言ってもらえれば Claude が流せるもの**(本番が変わるものは流す前に確認する)。「自分で」は画面の操作・パスワード・端末が要るもの。

#### 急ぎ(放っておくと困る)

| # | 作業 | 状態(実測) | どこで | 誰が |
|---|---|---|---|---|
| 1 | **平文の控えを消す** | **済(2026-09-18 実測)**: この PC の `D:\Backups\_opened\` は空、ホストの `km-before-*.dump`・`.env.bak-*` も無い | 03-backup / ホスト | 自分で |
| 2 | **Android のリリース版を ito4.jp で作り直して配る** | **作成済み(09-17 21:37、両フレーバー・署名あり)。配布と管理画面の APK 差し替えが残り。** `versionCode` は 1 のまま | [14](14-domain-ito4.ipynb) §1・§4-6 | 自分で(署名・配布) |
| 3 | **自分用の DB アカウントの権限を `Kosen_map` だけにする** | **済(09-18)**: `Ito@172.30.2.%` は `Kosen_map` だけ(SELECT・INSERT・UPDATE・DELETE・CREATE・DROP・INDEX・ALTER)。`Main` も `Kosen_map` だけ(09-18 実測) | §6-2 | — |
| 4 | **旧ドメインの証明書を certbot から消す** | **済(09-18)**: `live/ito8795.com` は `/etc/letsencrypt/Old/ito8795.com-20260917-213239/` へ退避 | [14](14-domain-ito4.ipynb) §6-2 | — |

#### 配備の続き(12 の未了分)

| # | 作業 | 状態(実測) | どこで | 誰が |
|---|---|---|---|---|
| 5 | **web と soketi の像を作り直す** | **済(09-18 実測)**: Soketi は Node **v24.21.0**、web の `pdo_pgsql` は **無し** | §2-3 | — |
| 6 | Logto を SUPERUSER でない利用者で繋ぐ | **済(09-18 実測)**: `LOGTO_DB_USER=logto_app`、Logto は `postgres://logto_app` で繋いでいる | §3 | — |
| 7 | Console(admin テナント)のパスワード方針と総当たりロック | **済(09-18 実測)**: admin・default とも 12 文字以上・流出パスワード拒否・利用者情報や連続文字を拒否。MFA は両テナント Mandatory | §4 | — |

#### 画面で確かめる

| # | 作業 | 状態 | どこで |
|---|---|---|---|
| 8 | Console のメール(SMTP)でテスト送信し、SPF・DKIM・DMARC が PASS | **済(09-18)**: 受信したヘッダで 3 つとも PASS | [14](14-domain-ito4.ipynb) §4-6 |
| 9 | reCAPTCHA の許可ドメインを `ito4.jp`・`admin.ito4.jp` に | **済(09-18、利用者が確認)** | [14](14-domain-ito4.ipynb) §5 |
| 10 | SSL Labs を `ito4.jp` でもう一度(TLS 1.2 に WEAK が無いこと) | **済(09-18)**: A+・WEAK なし | [14](14-domain-ito4.ipynb) §7 |
| 11 | DNS に CAA(`ito4.jp. CAA 0 issue "letsencrypt.org"`) | **済(09-18 実測)**: Cloudflare・Google の DoH でどちらも `0 issue "letsencrypt.org"` | DNS の業者の画面 |

#### この PC

| # | 作業 | 状態(実測) | どこで |
|---|---|---|---|
| 12 | `Test.zip` を控えの置き場へ移す | **済**: `D:\Backups\_archive\Test-20260712.zip`(本人と SYSTEM だけ) | §7-2 |
| 13 | SSH の鍵にパスフレーズを付け、ssh-agent に載せる | **済(09-18 実測)**: ssh-agent は自動起動で稼働、`km_vps`・`km_ops` はパスフレーズ付きで載っている(`km_backup` は門番で縛るので無し) | §7-3 |
| 14 | 控え専用の鍵の門番(§7-4 の A) | **済(2026-09-18 本番)**: `km_backup` を門番付きで登録・拒否の試験・実際の控え・週次タスクを `-Gated` で登録し直し | §7-4 |
| 17 | km の降格(§7-4 の B) | **済(2026-09-18 本番)**: kmops を作り置き場と門番を渡し、km を docker から外し、ubuntu を退役。km の鍵だけでは root に届かない | §7-4 B |

#### 決めること

| # | 作業 | メモ |
|---|---|---|
| 15 | `ito8795.com` のドメインを手放すか | **手放す**と決めた(09-17)。A レコードは削除済み。**登録そのものの解約が残り**(業者の画面) |
| 16 | 逆引き(PTR)を `mail.ito4.jp` に | **無視**と決めた(09-17)。いまは `ubuntu` |

#### 2026-09-17 に済んだもの

| 何を | 結果 |
|---|---|
| §2 の配備(網の分割・read_only・cap_drop・digest・src の :ro・`__Host-`) | 済(像の作り直しだけ残り = #5) |
| §5 管理画面の別オリジン | 済(`admin.ito4.jp`。[14](14-domain-ito4.ipynb) で当てた) |
| §6 MariaDB | 済: 自分用 `Ito@172.30.2.%`(TLS 必須)を作成、`Main` を `172.30.2.%`・TLS 必須に、`root@%` を削除、phpMyAdmin の root を閉じた(権限の範囲だけ残り = #3) |
| §7-1 `D:\Backups` の ACL | 済(本人と SYSTEM だけ・継承なし) |
| ドメインの統一([14](14-domain-ito4.ipynb)) | 済: 証明書・`.env`・Logto の戻り先と差出人・audience・旧い API リソースの削除・known_hosts・週次の控えのタスク |
| MFA | 壊れた設定を控えの値に戻し(両テナント Mandatory)、3 人とも ito4.jp で登録し直し済み |
| 管理ポートの許可 | `allow-admin-home.local.conf` にこの PC の回線 `60.112.5.32` |


### 本番で済ませたもの(2026-09-15)

| 何を | 結果 |
|---|---|
| 控えを取った(`backup-data.ps1`) | `D:\Backups\ito8795.com-20260915-094339`(暗号化済み)。MariaDB 33 表・Postgres 79 表・uploads 4 件。**テスト環境へ持っていくデータはこれ** |
| ランキングの利用者の行を消した | `Ito1111` の 1 行。場所と調べられた語は残した(控えは上) |
| Logto の 502 を直した(前日) | nginx の reload。再発しないよう upstream を `resolve` にした(手元) |

### 手元で済ませたもの(本番には未配備)

- compose: 網の分割・read_only・cap_drop・src の読み取り専用・全イメージの digest 固定・Logto の DB 利用者の差し替え口・管理画面の別オリジンの env
- nginx: 管理画面の別オリジン(`$km_host_role`)・6001 の `/apps/` を閉じる・Origin を `KM_ADMIN_URL` に・`allow 172.16.0.0/12` を外した
- PHP: セッション Cookie を `__Host-KMSID` に・`km_site_request_origin()`(サインインの戻り先を要求のホストで選ぶ)
- Dockerfile: web から pdo_pgsql を外し digest で固定 / Soketi を Node 24 に
- スクリプト: `logto-db-role.sql`・`host-cert.sh issue --also`・`host-setup.sh`(PHP の書く設定を先に作る)・`check-updates.sh`(タグ@digest)・`reset-app-ranking.php --all-users`
- Android: `build (1).gradle.kts` と未使用の 4 ファイルを `Test/Old/unused-20260915/` へ。両フレーバーとも試験 567 件・失敗 0、APK も作れた
- `check.php`: 989 件すべて通過(今回の変更の見張りを 40 件ほど足した)

### 使い捨ての環境で確かめたこと(本番のホストの上で。本番には触れていない)

| 何を | どう確かめたか | 結果 |
|---|---|---|
| Soketi を Node 24 に | フォークの `/app` を `node:24-trixie-slim` に載せ、read_only・cap_drop ALL・node 利用者で起動。公開と非公開チャンネルの購読・署名つき配信・違う鍵と署名なしの配信 | **今の Soketi と同じ結果**。メモリ 70〜106MiB |
| フォークが配るイメージ | `latest`・`2.0.0`・distroless | **どれも起動しない**(GLIBC 2.38 が要るのに土台が Debian 12) |
| Logto を SUPERUSER でない利用者で | 本番の Logto DB を使い捨ての Postgres に写し、`logto-db-role.sql` → Logto 1.43.0 を read_only・cap_drop ALL で起動 | 3001・3002・OIDC・サインイン設定が応答。権限の不足 0 件。`COPY … PROGRAM`・`pg_read_file`・ロール作成は断られる |
| Console のパスワード方針とロック | 上の写しに SQL を当てて Logto を再起動 | admin テナントの sign-in-exp に反映 |
| Postgres の read_only・cap_drop | 上の写しの Postgres | 起動・写し・問い合わせが通る |
| nginx の read_only・cap_drop | 使い捨ての nginx(本番と同じ digest) | 起動する |
| 管理画面の別オリジン | 使い捨ての nginx で「無効」と「有効」を 1 つずつ | 無効は今と同じ。有効は公開側の `/admin/` → 管理用へ 308、管理用で要らないパス → 公開側へ 308、ゲートのログイン先は管理用 |
| compose の解決 | `docker compose config` を本番のホストで(一時フォルダの写し) | 通る。網・read_only・cap_drop・digest・src のマウントが狙いどおり |

**まだ誰も確かめていないのは「全部そろえた通し」**(web と mariadb と phpmyadmin と mailserver の read_only・cap_drop、src の読み取り専用で画面の操作が通るか)。それが §1 のテスト環境の役目。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
php ..\src\scripts\check.php | Select-Object -Last 3

## 1. テスト環境で通す(次の段階)

**テスト環境は本番の写し。** 利用者が作る(2026-09-15 の指示)。**.env の読み書きは自由にしてよい。**

### 持っていくもの

| もの | 場所 |
|---|---|
| データ(MariaDB・Postgres・uploads・config・env) | `D:\Backups\ito8795.com-20260915-094339\km-backup-ubuntu-20260915-094338.tar.gz.cms`(開くのは [03-backup](03-backup.ipynb) の `open-backup.ps1`) |
| コード | このリポジトリ(`deploy-to-host.ps1 -HostName <テスト環境>`) |
| 復号の鍵 | `C:\Users\itota\.kosenmap\backup-private.pem`(**テスト環境へは持っていかない**) |

### 流れ

1. テスト環境を作る(**[13-local-env](13-local-env.ipynb)**。LAN の Ubuntu 機を `.env` の `KM_ENV=local` で立てる。Let's Encrypt も公開 DNS も使わない。本番の構成をそのまま持ち込むと、証明書が取れずに nginx と MariaDB が再起動を繰り返す)
2. 控えを戻す([03-backup](03-backup.ipynb) §8)。**`.env` は env.txt を元に、ドメインまわりだけテスト用に**
   - 本番の Logto の資格情報(アプリの ID と秘密)は DB ごと写るので、そのまま使える。戻り先の URL だけ [11](11-getting-started.ipynb) §6 の `logto-domain.php` でテスト用のドメインへ
   - **テスト環境から本物のメールを出さない。** `MAIL_ADMIN_TO` を自分だけにする(問い合わせの通知が本番の宛先へ飛ぶ)
3. `host-setup.sh --fix`(**up の前に。** PHP の書く設定と `cache/` が無いと Docker がディレクトリを作る)
4. §3 のセル(控えから戻すと Logto の表の持ち主が初期利用者に戻るので、**毎回流す**)
5. `docker compose build web soketi` → `docker compose up -d`
6. §2-4 の「通しの確認」と、下の画面の確認
7. 別オリジンも試すなら §5(テスト用の管理ドメインの DNS と証明書が要る)

### 画面で確かめること(テスト環境で全部)

- [ ] 公開の地図が出る。**教職員氏名の解除が効く**(セッションの `__Host-KMSID`)
- [ ] 管理画面にサインインできる(MFA まで)。**一度ログアウトされる**のは正常
- [ ] ファイル管理でアップロードして**落とせる**(`uploads/` が書ける)
- [ ] プロフィールの画像を変えられる(`uploads/`)
- [ ] 「アプリへ地図を配信する」でアクセスコードを作れる(`config/app-map.local.php` が書ける)
- [ ] 「地図データ公開設定」を保存できる(`config/map-access.local.php` が書ける)
- [ ] 「ダウンロード」で APK を差し替えられる(210MB。`/tmp` の上限 64m に当たらないか ——当たるなら `99-uploads.ini` の置き場を見直す)
- [ ] 管理画面のチャットと死活監視が「接続済み」になる(Soketi の Node 24)
- [ ] phpMyAdmin(8281)・Logto Console(3002)がゲートを通って開く
- [ ] 問い合わせを 1 通送り、自分に届く(mailserver が mail の網から送れる)
- [ ] Android の管理版と来場者版でサインインし、地図を取れる(テスト用のドメインでビルドしたもの)
- [ ] `host-backup.sh` が通る(root@localhost と POSTGRES_USER で取れる)
- [ ] `check-updates.sh` が「タグ@digest」を読み、Logto の版を正しく出す

### 持っていくデータを見る

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
$set = 'D:\Backups\ito8795.com-20260915-094339'
Get-ChildItem $set | Select-Object Name, Length, LastWriteTime | Format-Table -AutoSize
Get-Content (Join-Path $set 'summary.txt') -Encoding utf8

## 2. 配備する(網の分割・read_only・cap_drop・digest・src の読み取り専用・Soketi・`__Host-`・pdo_pgsql)

§7 の 4・8・9(別オリジン以外)・14。**まとめて 1 回の配備**で入る(compose を分けて当てると、網の名前が混ざる)。

| 変わること | 利用者から見えること |
|---|---|
| 網の名前が変わる(`devnet` → `edge`・`appdb`・`authdb`・`mail`) | **全コンテナが作り直される。数十秒サイトが止まる** |
| セッション Cookie の名前が `__Host-KMSID` になる | **全員が一度ログアウトされる**(公開ページの解除も閉じ直る) |
| web の PHP のセッションが tmpfs(`/tmp`)に移る | 以後、**web を作り直すたびに全員がログアウトされる**(以前もコンテナを作り直せば同じ) |
| Soketi が Node 24 になる | 見た目は変わらない。チャットと死活監視が繋ぎ直す |
| `allow 172.16.0.0/12` が消える | 管理系ポート(3002・8281・8025)に入れる回線が**許可ファイルの IP と SSH の `-D` だけ**になる。**先に 2-2 で自分の IP を足す** |
| 6001 の `/apps/` が 404 になる | ブラウザは使っていない(PHP はコンテナの中から直に叩く) |

**戻し方:** 手元の前の版(`Old/` か控えのコード)で `deploy-to-host.ps1 -Action up`。データは変わらないので、戻すのはファイルと compose だけ。
網を戻すと、また全コンテナが作り直される。

### 2-1. 下見(何が送られるか)

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\deploy-to-host.ps1 -WhatIfOnly

### 2-2. いま繋いでいる回線の IP を管理系ポートの許可に足す

`nginx/km/allow-admin-home.local.conf`(**ホストにだけ置く**。配備物に含めない)へ、SSH の接続元の IPv4 を足す。
**IP は画面に出さない**(このノートの出力が残るので)。回線が変わったら同じセルをもう一度。済んだあとの反映は 2-3 の up でされる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "いま繋いでいる回線の IP を、管理系ポート(3002・8281・8025)の許可に足します"
IP="${SSH_CLIENT%% *}"
F=nginx/km/allow-admin-home.local.conf
case "$IP" in
  *.*.*.*) ;;
  *) echo "IPv4 を読めませんでした(IPv6 で繋いでいる?)。足していません"; exit 1 ;;
esac
touch "$F"
if grep -qx "allow $IP;" "$F"; then
  echo "既に足してあります"
else
  printf '# %s に docs/12 §2-2 で足した\nallow %s;\n' "$(date +%F)" "$IP" >> "$F"
  echo "足しました(allow の行: $(grep -c '^allow ' "$F") 本)"
fi

### 2-3. 送って、ビルドして、立て直す

1. `-Action deploy` でファイルだけ置く(**`-Action up` はビルドしない** —— Soketi が Node 16 の像のまま起動してしまう)
2. `host-setup.ps1 -Fix` で、PHP の書く設定と `cache/` を先に作り、持ち主を揃える
3. web と soketi をビルドし、全体を `up -d`

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番へファイルを置きます(まだ立て直しません)"
.\deploy-to-host.ps1 -Action deploy -Yes

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "本番の持ち主と権限を揃え、PHP の書く設定と cache/ が無ければ作ります"
.\host-setup.ps1 -Fix

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "web と soketi をビルドし、全コンテナを新しい網で作り直します(数十秒止まる。全員が一度ログアウトされる)" --timeout 1800
docker compose build web soketi </dev/null
docker compose up -d </dev/null
docker compose ps --format 'table {{.Service}}\t{{.Status}}' </dev/null
docker network ls --format '{{.Name}}' | grep '^kosenmap_'

### 2-4. 通しの確認

健全さに加えて、**read_only・特権・網・書ける場所・拡張・Node・/apps/・Cookie の名前**を実物で見る。
`★` が出たら、その行の説明を読む。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
D="$(sed -n 's/^KM_DOMAIN=//p' .env | tail -n 1)"
echo "== コンテナ(read_only / 特権 / 網 / 健全さ)"
for c in $(docker compose ps -q); do
  docker inspect -f '{{.Name}}  ro={{.HostConfig.ReadonlyRootfs}}  drop={{.HostConfig.CapDrop}}  add={{len .HostConfig.CapAdd}}  nets={{range $k, $v := .NetworkSettings.Networks}}{{$k}} {{end}} health={{if .State.Health}}{{.State.Health.Status}}{{end}} oom={{.State.OOMKilled}}' "$c"
done | sed 's#^/##; s/kosenmap_//g' | sort
echo
echo "== web: src は読み取り専用、書く 4 か所だけ書ける(www-data として)"
docker compose exec -T -u www-data web sh -c 'for p in /var/www/html/index.php /var/www/html/config/db.local.php /var/www/html/uploads /var/www/html/cache /var/www/html/config/app-map.local.php /var/www/html/config/map-access.local.php; do if [ -w "$p" ]; then echo "  書ける   $p"; else echo "  書けない $p"; fi; done' </dev/null
echo "  (上 2 つが「書けない」、下 4 つが「書ける」なら狙いどおり)"
echo
echo "== web の PHP 拡張に pdo_pgsql が無いこと"
docker compose exec -T web php -m </dev/null | grep -qx pdo_pgsql && echo "  ★ まだ入っています(web をビルドし直していない)" || echo "  入っていません"
echo
echo "== Soketi の Node と利用者"
docker compose exec -T soketi node -e 'console.log("  Node", process.version, "uid", process.getuid())' </dev/null
echo
echo "== DB の網から外へ出られないこと(mariadb の中から)"
docker compose exec -T mariadb bash -c '(exec 3<>/dev/tcp/1.1.1.1/443) 2>/dev/null && echo "  ★ 外へ出られます(appdb が internal になっていない)" || echo "  外へは出られません"' </dev/null
echo
echo "== 入口"
printf '  6001 の /apps/ → '; curl -s -o /dev/null -m 10 -w '%{http_code}(404 が正しい)\n' "https://$D:6001/apps/x/events"
printf '  セッション Cookie の名前 → '; curl -s -m 10 -D - -o /dev/null "https://$D/admin/login.php" | sed -n 's/^[Ss]et-[Cc]ookie: \([^=]*\)=.*/\1/p' | head -n 1

### 2-5. 後片付け(古い網と、使わなくなった像)

`kosenmap_devnet` は誰も使わなくなる。像は **Node 16 の Soketi の土台**などが残る。
`docker image prune` は**どのコンテナも使っていない名無しの像**だけを消す(動いているものは消えない)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "使われなくなった網(kosenmap_devnet)と、名無しの像を片付けます"
docker network rm kosenmap_devnet 2>/dev/null && echo "kosenmap_devnet を消しました" || echo "kosenmap_devnet はありません(または使用中)"
docker image prune -f | tail -n 1
docker system df | head -n 3

## 3. Logto を SUPERUSER でない利用者で繋ぐ(§7 の 3)

**何が変わるか:** Logto が繋ぐ利用者が、Postgres の初期利用者(SUPERUSER)から `logto_app`(Logto の表の持ち主にしただけの利用者)に変わる。
Logto に SQL を差し込める穴が出ても、`COPY … TO PROGRAM` でコマンドを走らせたり、`pg_read_file` でファイルを読んだりはできない。

**どう動くか:** `scripts/logto-db-role.sql` の `KM-LOGTO-ROLE` の塊が、利用者を作り(あればパスワードを替え)、DB・スキーマ・表・連番・関数・型の持ち主を `logto_app` に移す。
Logto の表は RLS が有効だが、**持ち主は RLS を受けない**(FORCE の表は 0 個。あれば BYPASSRLS を足す)。セルはパスワードを英数字で作って `.env` に書き、Logto を立て直す。**パスワードは画面に出さない。**

- **§2 の配備の後に流す**(compose の `LOGTO_DB_USER` はそこで入る)
- **控えから戻したら、もう一度流す**(戻すと持ち主が初期利用者に戻る)
- まっさらな DB に Logto を初めて立てる(`db seed`)ときは CREATEROLE が要るので、**そのときだけ `.env` の 2 行を空にしておく**
- 控え(`host-backup.sh`)は今までどおり初期利用者で取る

**戻し方:** `.env` の `LOGTO_DB_USER` と `LOGTO_DB_PASSWORD` を空にして `docker compose up -d`。持ち主が `logto_app` のままでも、初期利用者は SUPERUSER なので読み書きできる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto 用の SUPERUSER でない利用者 logto_app を作り(あればパスワードを替え)、持ち物を移し、.env に書いて Logto を立て直します" --timeout 900
set -eu
PW="$(od -An -N24 -tx1 /dev/urandom | tr -d ' \n')"
{ printf "\\set pw '%s'\n" "$PW"; sed -n '/^-- KM-LOGTO-ROLE-BEGIN$/,/^-- KM-LOGTO-ROLE-END$/p' scripts/logto-db-role.sql; } \
  | docker compose exec -T postgres sh -c 'psql -U "$POSTGRES_USER" -d "$POSTGRES_DB" -q'
cp -p .env ".env.bak-$(date +%Y%m%d-%H%M%S)"
sed -i '/^LOGTO_DB_USER=/d;/^LOGTO_DB_PASSWORD=/d' .env
printf 'LOGTO_DB_USER=logto_app\nLOGTO_DB_PASSWORD=%s\n' "$PW" >> .env
unset PW
docker compose up -d </dev/null
i=0
until [ "$(docker inspect -f '{{.State.Health.Status}}' km-logto 2>/dev/null)" = healthy ] || [ "$i" -ge 240 ]; do i=$((i + 5)); sleep 5; done
echo "Logto: $(docker inspect -f '{{.State.Health.Status}}' km-logto)(${i} 秒)"

### 確かめる

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose exec -T logto sh -c 'echo "$DB_URL" | sed "s#://\([^:]*\):[^@]*@#://\1:<伏せ>@#"' </dev/null
docker compose exec -T postgres sh -c 'psql -U "$POSTGRES_USER" -d "$POSTGRES_DB" -At' <<'SQL'
select 'logto_app: super=' || rolsuper || ' createrole=' || rolcreaterole || ' createdb=' || rolcreatedb || ' bypassrls=' || rolbypassrls from pg_roles where rolname = 'logto_app';
select '表の持ち主: ' || tableowner || ' ' || count(*) || ' 個' from pg_tables where schemaname = 'public' group by tableowner;
SQL
docker compose logs --since 10m logto 2>&1 | grep -ciE 'permission denied|must be owner' | sed 's/^/権限の不足の記録: /'

## 4. Console(admin テナント)のパスワード方針と総当たりロック(§7 の 1 = A)

**MFA は 2026-09-15 の実測で既に Mandatory(Totp・WebAuthn)。** 残っていたのはパスワード方針(`{}`)と総当たりロック(`{}`)。
`KM-ADMIN-POLICY` の塊で default テナントと同じ値にする: **12〜256 文字・漏えい照合・利用者情報と連番を断る / 5 回失敗で 60 秒ロック。**
方針は**次にパスワードを決めるとき**から効く(今のパスワードはそのまま使える)。Logto は設定を覚えているので再起動する(数秒サインインが止まる)。

**戻し方:** Console の画面では admin テナントのこの値を変えられないので、SQL で `password_policy = '{}'`・`sentinel_policy = '{}'` に戻す。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Console(admin テナント)のパスワード方針と総当たりロックを default テナントと同じにし、Logto を再起動します" --timeout 600
sed -n '/^-- KM-ADMIN-POLICY-BEGIN$/,/^-- KM-ADMIN-POLICY-END$/p' scripts/logto-db-role.sql \
  | docker compose exec -T postgres sh -c 'psql -U "$POSTGRES_USER" -d "$POSTGRES_DB" -q -A -F " | "'
docker compose restart logto </dev/null

## 5. 管理画面を別オリジンに置く(§7 の 9)

> **2026-09-17: ドメインを ito4.jp へ移すことになったので、この §5 は [14-domain-ito4](14-domain-ito4.ipynb) の中で `admin.ito4.jp` として当てる。**ここのセル(`admin.ito8795.com`)は流さない。考え方と表はそのまま有効。

**なぜ:** 公開ページ(地図・問い合わせ・規約)と管理画面が同じオリジンだと、公開ページのどこか 1 か所にスクリプトを差し込まれた時点で、
そのスクリプトは管理画面を開いて CSRF トークンを読み、操作できる。**管理画面を別のホスト名に置けば、公開ページの穴は管理画面に届かない。**

**どうなるか**(`.env` に `KM_ADMIN_DOMAIN` を書いたときだけ。**書かなければ今までと同じ**):

| 開いた URL | 結果 |
|---|---|
| `https://ito8795.com/admin/…` | `https://<管理用>/admin/…` へ 308 |
| `https://<管理用>/admin/…`・`/sign-in.php`・`/callback.php`・`/Main/`・`/vendor-web/`・地図エディタの API | そのまま |
| `https://<管理用>/` やそのほかの公開ページ | `https://ito8795.com/…` へ 308 |
| `https://<管理用>:8281`・`:3002`・`:8025` | ゲート → 未ログインなら `https://<管理用>/admin/login.php` |
| Android アプリ・公開の API・Logto のサインイン(3001) | **変わらない**(公開側のまま) |

- 管理画面のサインインは管理用のホストで**別に**持つ(公開側でサインインしていても、管理用では入り直す。Logto のセッションが残っていれば MFA まで一瞬)
- **§2 の配備の後に**。DNS・証明書・Logto の登録の 3 つがそろってから `.env` を書く

**戻し方:** `.env` の `KM_ADMIN_DOMAIN` を消して `docker compose up -d`。Logto に足した URL は残しておいて害は無い。

### 5-1. DNS が向いているか

DNS の業者の画面で、管理用の名前(例 `admin.ito8795.com`)の **A レコードを `163.43.218.158`** に向けてから流す。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
ADMIN=admin.ito8795.com   # ← 管理用の名前
echo "このホスト: $(hostname -I | awk '{print $1}')"
echo "$ADMIN → $(getent ahostsv4 "$ADMIN" | awk '{print $1}' | sort -u | tr '\n' ' ')"

### 5-2. 証明書に管理用の名前を足す

証明書の名前は `KM_DOMAIN` のまま、**同じ証明書に管理用の名前を足す**(`--also`)。まず `--dry-run`(発行回数を消費しない)。
`MAIL` は Let's Encrypt から期限切れの警告を受け取る宛先。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "証明書に管理用の名前を足す発行を、試験用の発行元で試します(本物の証明書は変えません)" --timeout 600
ADMIN=admin.ito8795.com   # ← 管理用の名前
MAIL=admin@example.jp     # ← 期限切れの警告の宛先
./scripts/host-cert.sh issue --domain "$(sed -n 's/^KM_DOMAIN=//p' .env | tail -n 1)" --also "$ADMIN" --email "$MAIL" --dry-run </dev/null

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "証明書に管理用の名前を足して発行し直します(nginx は次の reload で拾います)" --timeout 600
ADMIN=admin.ito8795.com
MAIL=admin@example.jp
./scripts/host-cert.sh issue --domain "$(sed -n 's/^KM_DOMAIN=//p' .env | tail -n 1)" --also "$ADMIN" --email "$MAIL" </dev/null
docker compose exec -T reverse-proxy nginx -s reload </dev/null

### 5-3. Logto Console に管理用の戻り先を足す(画面で)

Console(`https://ito8795.com:3002`)→ アプリケーション → **管理画面の Web アプリ**(`.env` の `LOGTO_APP_ID` のもの):

| 欄 | 足す値(**公開側の分は消さない**) |
|---|---|
| リダイレクト URI | `https://admin.ito8795.com/callback.php` |
| サインアウト後のリダイレクト URI | `https://admin.ito8795.com/` |
| CORS の許可オリジン(欄があれば) | `https://admin.ito8795.com` |

**保存を押す。** 足していないと、管理用のホストでのサインインが `redirect_uri` の不一致で必ず落ちる。

### 5-4. 切り替える

Logto Console の URL も `https://<管理用>:3002` に変わる(Console の戻り先は Logto が作るので DB は変えない)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm ".env に KM_ADMIN_DOMAIN を書き、管理画面を別オリジンに切り替えます(nginx・web・logto・phpmyadmin が作り直されます)" --timeout 900
ADMIN=admin.ito8795.com   # ← 管理用の名前
set -eu
./scripts/host-cert.sh status </dev/null | grep -E "管理用の名前|★" || true
cp -p .env ".env.bak-$(date +%Y%m%d-%H%M%S)"
sed -i '/^KM_ADMIN_DOMAIN=/d' .env
printf 'KM_ADMIN_DOMAIN=%s\n' "$ADMIN" >> .env
docker compose up -d </dev/null
docker compose ps --format 'table {{.Service}}\t{{.Status}}' </dev/null

### 5-5. 確かめる

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
D="$(sed -n 's/^KM_DOMAIN=//p' .env | tail -n 1)"
A="$(sed -n 's/^KM_ADMIN_DOMAIN=//p' .env | tail -n 1)"
for u in "https://$D/admin/" "https://$A/admin/login.php" "https://$A/" "https://$A:8281/" "https://$D/"; do
  printf '%-45s → ' "$u"; curl -s -o /dev/null -m 10 -w '%{http_code} %{redirect_url}\n' "$u"
done
echo "(公開の /admin/ は 308 で管理用へ / 管理用の login.php は 200 / 管理用の / は 308 で公開側へ / 8281 は 302 で管理用のログインへ / 公開の / は 200)"

## 6. MariaDB: `root@%` と `Main@%`、自分だけの最低権限アカウント(§7 の 2)

**§2 の配備の後に**(`appdb` の網の範囲 `172.30.2.0/24` が本番にできてから)。順番:

1. **6-1** phpMyAdmin の root を一時的に開ける
2. **6-2** 自分で phpMyAdmin に root で入り、**自分だけのアカウントを作る**(パスワードは自分で決める。このノートにもチャットにも書かない)
3. **6-3** `Main` を `appdb` の網からだけにし、`root@%` を消し、phpMyAdmin の root を閉じる

**`root@localhost` は残る**(コンテナの中からの控え `host-backup.sh` と、このノートのセルが使う)。外から root で入る口は無くなる。

### 6-1. phpMyAdmin の root を一時的に開ける

phpMyAdmin は **`172.30.2.x`(appdb の網)から** DB に繋ぐ。6-3 で `root@%` を消したあとは、網から入れる root が居ないので、
**`KM_PMA_ALLOW_ROOT=1` だけでは root で入れない**(2026-09-18 に「root でログインできない」と報告。前の版のこのセルはそれだけだった)。

そこで、**`root@localhost` と同じパスワードの `root@'172.30.2.%'`(TLS 必須)を一時的に作ってから**開ける。
パスワードは `root@localhost` の**ハッシュを写す**だけで、値そのものはセルにも出力にも出ない。**6-3 で両方とも閉じる。**
何度流してもよい(在れば作らない)。2026-09-18 に検証機で、作れる・2 回流しても 1 つ・TLS 無しでは断られる、を確かめた。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "phpMyAdmin に root で入れるようにします(一時的。用が済んだら 6-3 で必ず閉じる)" --timeout 600
set -eu
docker compose exec -T mariadb sh -c 'MYSQL_PWD="$MARIADB_ROOT_PASSWORD" exec mariadb -uroot' <<'SQL'
SET @h = (SELECT JSON_VALUE(Priv, '$.authentication_string') FROM mysql.global_priv WHERE User = 'root' AND Host = 'localhost');
SET @s = CONCAT('CREATE USER IF NOT EXISTS ''root''@''172.30.2.%'' IDENTIFIED BY PASSWORD ''', @h, ''' REQUIRE SSL');
PREPARE st FROM @s; EXECUTE st; DEALLOCATE PREPARE st;
GRANT ALL PRIVILEGES ON *.* TO 'root'@'172.30.2.%' WITH GRANT OPTION;
SELECT user, host, ssl_type,
       IF(authentication_string = (SELECT authentication_string FROM mysql.user WHERE user = 'root' AND host = 'localhost'), 'localhost と同じ', '★ 違う') AS password
  FROM mysql.user WHERE user = 'root' ORDER BY host;
SQL
cp -p .env ".env.bak-$(date +%Y%m%d-%H%M%S)"
sed -i '/^KM_PMA_ALLOW_ROOT=/d' .env
printf 'KM_PMA_ALLOW_ROOT=1\n' >> .env
docker compose up -d phpmyadmin </dev/null
echo "phpMyAdmin の root: $(docker compose exec -T phpmyadmin printenv KM_PMA_ALLOW_ROOT </dev/null | tr -d '\r')(1 で開いている)"


### 6-2. 自分のアカウントを作る(phpMyAdmin の画面で)

root のパスワードは別の窓で見る(**ノートの出力に残さないため**)。窓を閉じれば消える。

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_ops" kmops@ito4.jp "grep '^MARIADB_ROOT_PASSWORD=' /opt/kosenmap/.env; read -r -p 'Enter で閉じる' _"

1. `https://admin.ito4.jp:8281`を開き、**ユーザー名 `root`** と上のパスワードで入る
2. 上の「SQL」タブで、**名前とパスワードを自分のものに書き換えて**実行する:

```sql
CREATE USER 'ここに自分の名前'@'172.30.2.%' IDENTIFIED BY 'ここに自分で決めたパスワード(16 文字以上)' REQUIRE SSL;
GRANT SELECT, INSERT, UPDATE, DELETE ON `Kosen_map`.* TO 'ここに自分の名前'@'172.30.2.%';
```

> **phpMyAdmin の「ユーザアカウント」→「グローバル権限」でチェックを付けない。** そこで付けると `*.*`(**`mysql` を含む全 DB**)に付き、
> 自分の権限を書き換えられる = 実質 root になる(2026-09-17 に実際にそうなった)。付けるなら上の SQL か、「データベース」タブで `Kosen_map` を選んでから。
> 表の形も変えるなら `GRANT CREATE, ALTER, INDEX, DROP ON \`Kosen_map\`.* TO …` を足す。

3. ログアウトし、**作ったアカウントで入り直せる**ことを確かめる。`Kosen_map` の表が見えて、行を直せれば足りている

| このアカウントで | できる / できない |
|---|---|
| `Kosen_map` の行を見る・足す・直す・消す | できる |
| 表を作る・消す・列を変える(CREATE / DROP / ALTER / INDEX) | 上の SQL だけなら**できない**。2026-09-18 に `Ito` へは `Kosen_map` に限って付けた |
| ほかの DB(`mysql` など)・利用者の追加・権限の付与(GRANT)・ファイルの読み書き(FILE) | **できない** |
| phpMyAdmin の外(ホストや別のコンテナ)から繋ぐ | **できない**(`172.30.2.%` = appdb の網からだけ。phpMyAdmin はここに居る) |

### 6-3. `Main` を網からだけにし、`root@%` を消し、root を閉じる

**自分のアカウントで入れたのを確かめてから。** 先に web と phpMyAdmin が `172.30.2.x` から繋いでいることを見る。
`Main` には TLS も必須にする(web は既に TLS で繋いでいる)。最後に web から DB を 1 回読んで、繋がることを見る。

**何度流してもよい。** `Main@%` が在るときだけ移す(前の版は 2 回目に「無い利用者の RENAME」で落ちた)。
6-1 で作った `root@'172.30.2.%'` も消す。出力のパスワードのハッシュは伏せる。2026-09-18 に検証機で、未移行・移し済み・本物の利用者の 3 通りを流して確かめた。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "アプリの利用者を appdb の網(172.30.2.%)からだけにして TLS を必須にし、root@% と root@172.30.2.% を消し、phpMyAdmin の root を閉じます" --timeout 600
set -eu
for c in km-php-apache km-phpmyadmin; do
  docker inspect -f '{{.Name}} の appdb の住所: {{with index .NetworkSettings.Networks "kosenmap_appdb"}}{{.IPAddress}}{{else}}★ appdb に居ません(§2 の配備が済んでいない){{end}}' "$c"
done
U="$(docker compose exec -T mariadb printenv MARIADB_USER </dev/null | tr -d '\r')"
km_sql() { docker compose exec -T mariadb sh -c 'MYSQL_PWD="$MARIADB_ROOT_PASSWORD" exec mariadb -uroot -N'; }
# 何度流してもよいように、今の形を見てから決める(2026-09-17、2 回目で RENAME が「無い利用者」で落ちた)
has_any="$(printf "SELECT COUNT(*) FROM mysql.user WHERE user = '%s' AND host = '%%';\n" "$U" | km_sql | tr -d '\r')"
has_net="$(printf "SELECT COUNT(*) FROM mysql.user WHERE user = '%s' AND host = '172.30.2.%%';\n" "$U" | km_sql | tr -d '\r')"
RENAME=""
if [ "$has_any" = 1 ] && [ "$has_net" = 1 ]; then
  echo "★ $U@% と $U@172.30.2.% が両方あります。どちらを残すか決めてから流してください"; exit 1
elif [ "$has_any" = 1 ]; then
  RENAME="RENAME USER '$U'@'%' TO '$U'@'172.30.2.%';"
  echo "$U@% を 172.30.2.% へ移します"
elif [ "$has_net" = 1 ]; then
  echo "$U は移し済みです(172.30.2.%)"
else
  echo "★ $U が見つかりません"; exit 1
fi
docker compose exec -T mariadb sh -c 'MYSQL_PWD="$MARIADB_ROOT_PASSWORD" exec mariadb -uroot' <<SQL | sed "s/IDENTIFIED BY PASSWORD '[^']*'/IDENTIFIED BY PASSWORD '<伏せ>'/"
$RENAME
ALTER USER '$U'@'172.30.2.%' REQUIRE SSL;
DROP USER IF EXISTS 'root'@'%';
DROP USER IF EXISTS 'root'@'172.30.2.%';
SELECT user, host, ssl_type FROM mysql.user ORDER BY user, host;
SHOW GRANTS FOR '$U'@'172.30.2.%';
SQL
cp -p .env ".env.bak-$(date +%Y%m%d-%H%M%S)"
sed -i '/^KM_PMA_ALLOW_ROOT=/d' .env
printf 'KM_PMA_ALLOW_ROOT=\n' >> .env
docker compose up -d </dev/null
docker compose exec -T web php -r 'require "/var/www/html/lib/db.php"; echo "web から DB: 地点 ", km_db()->query("SELECT COUNT(*) FROM km_map_nodes")->fetchColumn(), " 件\n";' </dev/null


**戻し方:** root で入れない状態なので、コンテナの中から root@localhost で戻す(`RENAME USER '<利用者>'@'172.30.2.%' TO '<利用者>'@'%'`)。
網を戻す(§2 の戻し)ときは、**先にこれを戻す**(戻さないと web の住所が 172.30.2.x でなくなり、DB に入れない)。

## 7. この PC(§7 の 5・10・11)

### 7-1. D:\Backups を本人と SYSTEM だけにする

いまは `D:\` から継いだ **Authenticated Users が変更できる**(同じ PC の別の利用者が、開いた控えを読める・消せる)。
継承を切って、**本人(フル)と SYSTEM(フル)だけ**にする。今の ACL は先に控える(`icacls /restore` で戻せる)。
週次の控えのタスク(`KosenMap バックアップ(週次)`)は本人で動くので、そのまま動く。

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps
$root = 'D:\Backups'
$saved = Join-Path $env:USERPROFILE ".kosenmap\Backups-acl-before-$(Get-Date -Format yyyyMMdd-HHmmss).txt"
icacls $root /save $saved /t /q | Out-Null
"今の ACL の控え: $saved(戻すときは icacls D:\ /restore <この控え>)"
icacls $root /inheritance:r /grant:r "${env:USERNAME}:(OI)(CI)F" '*S-1-5-18:(OI)(CI)F'
icacls $root
"--- 中身も継いでいるか(1 つめの世代)"
Get-ChildItem $root -Directory | Select-Object -First 1 | ForEach-Object { icacls $_.FullName }

### 7-2. Test.zip を控えの置き場へ移す(7-1 の後)

`Test/Test.zip`(2026-07-12)には **DB の接続設定(パスワードらしき項目)と local.properties** が入っている。
共有やアップロードで一緒に出ていかないよう、ACL を絞った `D:\Backups\_archive\` へ移す。**`_` で始まるフォルダは控えの世代整理が数えない**ので消されない。

**中の DB の資格情報が今も使われているか**は、開いて本番の `.env`(`MARIADB_PASSWORD`)と見比べる。同じなら、そのパスワードは替える(別の作業)。

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。

In [ ]:
%%ps
$src = 'C:\Users\itota\Documents\Test\Test.zip'
$dir = 'D:\Backups\_archive'
if (-not (Test-Path $src)) { "もうありません: $src"; return }
New-Item -ItemType Directory -Force $dir | Out-Null
$dest = Join-Path $dir ('Test-{0:yyyyMMdd}.zip' -f (Get-Item $src).LastWriteTime)
Move-Item -LiteralPath $src -Destination $dest
Get-Item $dest | Select-Object FullName, Length, LastWriteTime
(Get-Acl $dest).Access | Select-Object IdentityReference, FileSystemRights, IsInherited | Format-Table -AutoSize

### 7-3. SSH の鍵にパスフレーズを付け、ssh-agent に載せる(§7 の 11)

**km は docker グループに居るので、ホストの上では実質 root。** その鍵がこの PC に**パスフレーズ無しで**置いてあるので、
**この PC が破られるとホストの root まで届く。** パスフレーズを付け、ssh-agent(Windows のサービス)に載せる。
agent に載せておけば、`%%host` のセルも週次の控えのタスク(本人で動く)も今までどおり通る。**再起動しても agent は鍵を覚えている。**

1. 別の窓でパスフレーズを付ける(**パスフレーズは自分で決めて、どこにも書かない**)
2. 管理者の窓で ssh-agent を自動起動にする(サービスの設定なので管理者が要る)
3. 別の窓で鍵を agent に載せる
4. 下の 🟢 のセルで、パスフレーズを聞かれずに繋がるかを見る

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh-keygen -p -f "$env:USERPROFILE\.ssh\km_vps"

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
Start-Process pwsh -Verb RunAs -ArgumentList '-NoExit', '-Command', 'Set-Service ssh-agent -StartupType Automatic; Start-Service ssh-agent; Get-Service ssh-agent'

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh-add "$env:USERPROFILE\.ssh\km_vps"; ssh-add -l

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
ssh-add -l
ssh -o BatchMode=yes -o ConnectTimeout=10 -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "echo 'パスフレーズを聞かれずに繋がりました'"
"パスフレーズが付いているか: " + $(if ((ssh-keygen -y -P '' -f "$env:USERPROFILE\.ssh\km_vps" 2>&1) -match 'incorrect passphrase|load failed') { '付いている' } else { '★ 付いていない' })
# 上の ssh-keygen -y は「付いている」ときに 255 で終わる。それはこのセルの失敗ではない
$global:LASTEXITCODE = 0


### 7-4. km の降格

パスフレーズは「この PC が破られたとき」の守りで、**km そのものが実質 root である点は変わらない。** 降格の案:

| 案 | 何をするか | 効き | 手間・影響 | 状態 |
|---|---|---|---|---|
| **A. 控え専用の鍵を分ける** | 週次の控えだけに使う鍵 `km_backup` を作り、`authorized_keys` で `restrict,command="…/ssh-backup-gate.sh"` に縛る | 無人で動く鍵が**控えを取る以外に使えない** | 小 | **済(2026-09-18 本番)** |
| **B. 配備と日々の作業を分ける** | docker を使う作業を専用の利用者 `kmops` へ移し、`km` を docker から外す | km の鍵が盗まれても、sudo のパスワード無しには root に届かない | 中 | **済(2026-09-18 本番。段 1〜4)** |
| C. rootless Docker | Docker を利用者権限で動かす | docker の破りが root に直結しない | 大。80/443 の待ち受け・ボリュームの持ち主・cron の作り直し | 見送り |

#### A の結果(2026-09-18)

- 本番の `~km/.ssh/authorized_keys` に 1 行足した(控え: `authorized_keys.bak-20260918-001005`)
- この鍵で **ping・backup・fetch・report-failure だけが通り**、シェル・`id`・`;` でつなげる・知らない引数・`../` を含む名前・`.env` の scp(SCP/SFTP とも)・書き込み・ポート転送(`administratively prohibited`)は断られた
- 門番を通して本当に控えを取った: `D:\Backups\ito4.jp-20260918-001427`(MariaDB 33 表・Postgres 79 表・uploads 4)
- 週次のタスクを `register-backup-task.ps1 -Gated` で登録し直した(鍵 `km_backup`)


#### B. km の降格 —— 段取り

**本番で読んだこと(2026-09-18):**

- 定期の仕事(証明書・控え・更新・セキュリティ確認)は **root の cron** で動く → km を docker から外しても止まらない
- km は `sudo`(**パスワードが要る**)と `docker`(**要らない**)の両方に居る。docker が裏口になっている
- **`ubuntu`(クラウドの既定の利用者)も docker と sudo に居て、`AllowUsers ubuntu` で SSH が開いている**(最後に入ったのは 08-28)
- docker を使うのは: 配備(`deploy-to-host.ps1`)・`%%host` のセル・門番の `backup`。sudo を使うのは `ssh -t … sudo host-*.sh` のセル

**役の分け方:**

| 利用者 | 入る鍵 | docker | sudo | 使いみち |
|---|---|---|---|---|
| `km` | `km_vps` | **外す** | パスワード必須 | 人が入って `sudo host-*.sh`・`sudo reboot` |
| `kmops` | `km_ops`(**パスフレーズ付き**・転送を切る・送信元を絞れる) | 入る | **入れない** | 配備・`%%host`・門番の控え |
| `ubuntu` | ― | 外す | 外す | 使わない(`AllowUsers` から外す) |

**正直な限界:** kmops の鍵は依然として「docker = root」に届く。効くのは、**その鍵をパスフレーズ付き・この PC だけ・(できれば)送信元の IP 付きにして、普段使う km の鍵とは別にしておく**ところ。

**段(1 つずつ。段ごとに検証機 → 本番の順で、配備・`%%host`・控えが通るのを見てから次へ):**

| 段 | 何をする | 戻し方 | 状態 |
|---|---|---|---|
| 1 | `host-ops-user.sh create`: kmops を作り、docker に入れ、鍵を置き、`AllowUsers` に足す。**km には触れない** | `sudo userdel -r kmops`・`99-km.conf` を `.bak-日時` に戻す | **済(09-18 検証機・本番)** |
| 2 | `host-ops-user.sh handover`: `/opt/kosenmap` の中で km が持つものを kmops へ、門番の鍵の行を km から kmops へ。PC の配備・`%%host`・控えを kmops に切り替える | `sudo host-ops-user.sh handback`(持ち主の控えは `/var/backups/kosenmap/owners-*.tsv`) | **済(09-18 検証機・本番)** |
| 3 | `host-ops-user.sh demote`: 引き継ぎ(持ち主・門番の行・kmops の docker・km の sudo)を確かめてから km を docker から外す | `sudo host-ops-user.sh undemote` | **済(09-18 検証機・本番)** |
| 4 | `host-ops-user.sh retire`: ubuntu を docker・sudo・adm から外し、`AllowUsers` から外し、ログインを閉じ、鍵を退避する(**消さない**) | `sudo host-ops-user.sh unretire --record /var/backups/kosenmap/retire-ubuntu-日時.txt` | **済(09-18 検証機・本番)** |


##### 段 1-1. kmops 用の鍵を作る(この PC)

**パスフレーズは自分で決めて、どこにも書かない。** コメントは英数字と `@ . _ -` だけにする(スクリプトがそれ以外を断る)。

🔑 **別の窓で開く** —— パスフレーズの入力が要ります。


In [ ]:
%%terminal
$key = "$env:USERPROFILE\.ssh\km_ops"
if (Test-Path $key) { Write-Host "既にあります: $key(作り直しません)" } else { ssh-keygen -t ed25519 -f $key -C "kmops@$env:COMPUTERNAME" }
ssh-keygen -l -f "$key.pub"


##### 段 1-2. 鍵を ssh-agent に載せる

agent が止まっていたら、先に §7-3 の 2 つ目のセル(管理者の窓で自動起動にする)を実行する。**載せておくと、配備や `%%host` がパスフレーズを聞かずに通る。**

🔑 **別の窓で開く** —— パスフレーズの入力が要ります。


In [ ]:
%%terminal
Get-Service ssh-agent
ssh-add "$env:USERPROFILE\.ssh\km_ops"; ssh-add -l

##### 段 1-3. 検証機で kmops を作る

km の sudo のパスワードを聞かれる。**本番ではなく検証機(192.168.217.128)。** `--from` は付けない(検証機は LAN の中)。

🔑 **別の窓で開く** —— sudo のパスワードが要ります。


In [ ]:
%%terminal
$pub = (Get-Content "$env:USERPROFILE\.ssh\km_ops.pub" -Raw).Trim()
ssh -t -i "$env:USERPROFILE\.ssh\test" km@192.168.217.128 -o IdentitiesOnly=yes "sudo /opt/kosenmap/scripts/host-ops-user.sh create --pubkey-line '$pub'"


##### 段 1-4. 検証機で確かめる

- kmops で入れて、docker が使え、sudo が使えない
- **km は今までどおり**(`status` が読める)

🟢 **読むだけ** —— 何も変えません。


In [ ]:
%%ps
$vm = '192.168.217.128'
$ops = @('-o', 'BatchMode=yes', '-o', 'IdentitiesOnly=yes', '-o', 'ConnectTimeout=10', '-i', "$env:USERPROFILE\.ssh\km_ops")
'--- kmops で入る'
ssh @ops "kmops@$vm" "id; docker ps --format '{{.Names}}' | head -n 3; sudo -n true 2>&1 | head -n 1"
'--- 状態(km で読む)'
ssh -o BatchMode=yes -i "$env:USERPROFILE\.ssh\test" "km@$vm" "/opt/kosenmap/scripts/host-ops-user.sh status"


**検証機の結果(2026-09-18):** kmops で入れて docker が使え、`sudo` は `a password is required` で断られた。
`AllowUsers` は `km kmops`、km は変わらず docker に居る(段 3 まで)。ポート転送は `administratively prohibited`、agent の転送も届かない。

##### 段 1-5. 本番で kmops を作る

**本番の sshd を読み直す**(今つながっている SSH は切れない)。書き換える前に `99-km.conf.bak-日時` を取り、`sshd -t` が通らなければ戻して止まる。

`--from 60.112.5.32` で、**kmops の鍵はこの回線からしか使えない**ようにする(管理ポートの許可と同じ回線)。
回線の IP が変わったら kmops では入れなくなるが、**km はこれまでどおり入れる**ので、km で入って `--from` を新しい IP にして同じセルを流し直せば戻る。

🔑 **別の窓で開く** —— sudo のパスワードが要ります。


In [ ]:
%%terminal
$pub = (Get-Content "$env:USERPROFILE\.ssh\km_ops.pub" -Raw).Trim()
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-ops-user.sh create --from 60.112.5.32 --pubkey-line '$pub'"


##### 段 1-6. 本番で確かめる

🟢 **読むだけ** —— 何も変えません。


In [ ]:
%%ps
$ops = @('-o', 'BatchMode=yes', '-o', 'IdentitiesOnly=yes', '-o', 'ConnectTimeout=10', '-i', "$env:USERPROFILE\.ssh\km_ops")
'--- kmops で入る'
ssh @ops kmops@ito4.jp "id; docker ps --format '{{.Names}}' | head -n 3; sudo -n true 2>&1 | head -n 1"
'--- 状態(km で読む)'
ssh -o BatchMode=yes -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "/opt/kosenmap/scripts/host-ops-user.sh status"


**本番の結果(2026-09-18):** kmops(uid 1002)で入れて docker が使え、`sudo` は断られた。`AllowUsers` は `km kmops ubuntu`。km は変わらない。

#### 段 2. 置き場と鍵を kmops へ渡す

**何が変わるか:**

- `/opt/kosenmap` の中で **km が持つファイルの持ち主を kmops に**変える(モードは変えない。`.env` は kmops だけが読める 600 のまま)。
  変える前の持ち主は `/var/backups/kosenmap/owners-handover-日時.tsv` に控える
- **控え専用の鍵(門番)の行を km から kmops へ移す。** 段 3 で km を docker から外すと、km の権限で動く門番の `backup` が docker を使えなくなるため
- **km はまだ docker に居る。** 段 2 の後は、km では置き場に書けない(配備は kmops で行う)

**戻し方:** `sudo /opt/kosenmap/scripts/host-ops-user.sh handback`(持ち主と門番の行を km に戻す)

##### 段 2-1. 検証機で渡す

🔑 **別の窓で開く** —— sudo のパスワードが要ります。


In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\test" km@192.168.217.128 "sudo /opt/kosenmap/scripts/host-ops-user.sh handover"


**検証機の結果(2026-09-18):** 持ち主の控え 1075 件。`/opt/kosenmap`・`.env`(600)は kmops に、km が持つものは 0 件、門番の行は kmops へ。
そのあと kmops と `km_ops` の鍵で次が通った:

| 何を | 結果 |
|---|---|
| kmops で置き場に書く・`.env` を読む | 通る。**km では `Permission denied`**(狙いどおり) |
| 配備(`deploy-to-host.ps1 -User kmops`) | 通る。後片付けの自己検査 761 件・気になる点 0 |
| `%%host`(`km_nb` を kmops で) | 通る(`docker compose ps`・web から DB) |
| 門番(`km_backup` の鍵で kmops へ)の ping・backup | 通る。**km へは `Permission denied`**(行が移った) |

**途中で見つけたこと:** 検証機の `uploads/` の画像はグループが `root` で(控えから戻したときの名残)、**kmops でも km でも読めず、控えが `Permission denied` で落ちた。**
ディレクトリだけを見る後片付けの確認は OK と言っていた。`host-setup.sh` が**中のファイルが読めるか**も見て、`--fix` でグループを直すようにした(本番の画像は `www-data:km` なので handover で kmops になる)。

##### 段 2-2. 本番で渡す(PC の切り替えと一緒に)

**PC 側の既定は kmops に切り替え済み**(`deploy-to-host.ps1`・`host-setup.ps1`・`backup-*.ps1`・`register-backup-task.ps1`・`km_nb` の `%%host`)。
**このセルを流すまで、本番への配備と `%%host` は通らない**(kmops は置き場に書けない)。間を空けずに流す。

🔑 **別の窓で開く** —— sudo のパスワードが要ります。


In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-ops-user.sh handover"


##### 段 2-3. 週次の控えのタスクを kmops で登録し直す

**門番の行が kmops へ移るので、km で登録したままのタスクは次の日曜に失敗する。** handover のすぐ後に。

🟡 **手元が変わる** —— この PC にファイルを作る・消す、登録するなど。本番には触れません。


In [ ]:
%%ps
.\register-backup-task.ps1 -Gated
(Get-ScheduledTask -TaskName 'KosenMap バックアップ(週次)').Actions[0].Arguments


**本番の結果(2026-09-18):** 置き場・`.env`(600)・`backups/`(2770)・`uploads/` は kmops に、km の持ち物は 0 件、読めないファイル 0 件。門番の行は kmops へ。km では書けない。
週次のタスクは `-User "kmops" -KeyPath km_backup -Gated` で登録し直した。そのあと既定(kmops)のまま:

| 何を | 結果 |
|---|---|
| 配備 | 通る(自己検査 761 件・気になる点 0。所有者の値 gid=1002 = kmops) |
| `%%host` | 通る(9 つのコンテナが healthy・web から DB) |
| 週次と同じ手順の控え(`backup-task-run.ps1 -Gated`) | **1 回目は失敗**、直して 2 回目は通った(`ito4.jp-20260918-020229`) |

**1 回目の失敗:** 同じ時刻に週次のタスクも手で起動されていて、**先に終わった方の世代整理が、もう一方の「取得中でまだ空の保存先」を空の世代として消した。**
後の方が「取得に失敗しました」で落ち、**失敗の知らせのメールが出た**(実際には控えは取れている = 誤報)。
直したこと: `backup-data.ps1` が PC 全体のロック(名前付きミューテックス)を取り、重なったら待つ。世代整理は作って 2 時間以内の空の保存先を消さない。

#### 段 3. km を docker から外す

`demote` は**先に引き継ぎを全部見て、1 つでも欠けていたら何も変えない**: kmops が docker に居る・置き場の持ち主が kmops・置き場に km の持ち物が無い・門番の行が km に無い・km が sudo に居る(外したあともホストを直せる)。

**外したあとも変わらないもの:** `ssh -t km@… "sudo /opt/kosenmap/scripts/host-*.sh"` のセル(root で動く)、定期の仕事(root の cron)。
**変わるもの:** km で入って `docker` を直に打つことはできなくなる(打つなら kmops か `sudo docker`)。

**いま開いている km のセッションは docker を使えるまま**(グループはログインのときに決まる)。確かめは入り直してから。

##### 段 3-1. 検証機で外す

🔑 **別の窓で開く** —— sudo のパスワードが要ります。


In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\test" km@192.168.217.128 "sudo /opt/kosenmap/scripts/host-ops-user.sh demote"


##### 段 3-2. 検証機で確かめる

- km(入り直した新しいセッション)で `docker ps` が**断られる**。`sudo -n true` もパスワードを求める(= km の鍵だけでは root に届かない)
- kmops で配備と `%%host` が今までどおり通る(この PC の既定は本番なので、ここは検証機を明示する)

🟢 **読むだけ** —— 何も変えません。


In [ ]:
%%ps
$vm = '192.168.217.128'
'--- km(新しいセッション)'
ssh -o BatchMode=yes -i "$env:USERPROFILE\.ssh\test" "km@$vm" "id; docker ps 2>&1 | head -n 1; sudo -n true 2>&1 | head -n 1"
'--- kmops'
ssh -o BatchMode=yes -o IdentitiesOnly=yes -i "$env:USERPROFILE\.ssh\km_ops" "kmops@$vm" "id -un; docker ps --format '{{.Names}}' | wc -l"
'--- 状態'
ssh -o BatchMode=yes -i "$env:USERPROFILE\.ssh\test" "km@$vm" "/opt/kosenmap/scripts/host-ops-user.sh status"


**検証機の結果(2026-09-18):** km(入り直したセッション)の `docker ps` は `permission denied`、`sudo -n` は `a password is required`。docker に居るのは kmops だけ。kmops の docker・門番の ping は今までどおり。

**別に見つけたこと:** 検証機の km は **`lxd` グループにも居る**(Ubuntu の初期設定で最初の利用者が入る)。lxd も docker と同じく root に届く。
本番の km は lxd に居ない(本番で `docker lxd lxc sudo adm disk shadow libvirt` の顔ぶれを見た)。`host-ops-user.sh` の `status` と `demote` が、こうしたグループを「注意」で出すようにした(外すのは人が決める)。
検証機で外すなら: `ssh -t -i ~/.ssh/test km@192.168.217.128 "sudo gpasswd -d km lxd"`(本番と同じ形にそろう)。

##### 段 3-3. 本番で外す

**外したあと km では docker を直に打てなくなる。** host-*.sh を sudo で走らせるセル・定期の仕事(root の cron)・配備・`%%host`・週次の控え(kmops)は変わらない。

🔑 **別の窓で開く** —— sudo のパスワードが要ります。


In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-ops-user.sh demote"


##### 段 3-4. 本番で確かめる

🟢 **読むだけ** —— 何も変えません。


In [ ]:
%%ps
'--- km(新しいセッション)'
ssh -o BatchMode=yes -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "id; docker ps 2>&1 | head -n 1; sudo -n true 2>&1 | head -n 1"
'--- kmops'
ssh -o BatchMode=yes -o IdentitiesOnly=yes -i "$env:USERPROFILE\.ssh\km_ops" kmops@ito4.jp "id -un; docker ps --format '{{.Names}}' | wc -l"
'--- 門番'
ssh -o BatchMode=yes -o IdentitiesOnly=yes -i "$env:USERPROFILE\.ssh\km_backup" kmops@ito4.jp ping
'--- 状態'
ssh -o BatchMode=yes -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "/opt/kosenmap/scripts/host-ops-user.sh status"


**本番の結果(2026-09-18):** km(入り直したセッション)は `groups=km,sudo,users`、`docker ps` は `permission denied`、`sudo -n` は `a password is required`。
kmops は docker(9 コンテナ)、門番は `km-ok`。docker に居るのは `ubuntu,kmops`。**km の鍵(`km_vps`)だけでは、もう root に届かない。**

#### 段 4. ubuntu を退役させる

本番の `ubuntu` はクラウドの最初の利用者で、**docker と sudo に居て `AllowUsers ubuntu` で SSH が開いている**(最後に入ったのは 08-28)。使っていないのに、ここが root への経路として残っている。

`retire` がすること(**利用者もホームも鍵も消さない**):

1. 先に確かめる: いま `ubuntu` で sudo していない・km が sudo と SSH を持つ・sudoers に `ubuntu` の個別の行が無い。欠けたら何も変えない
2. 戻すための記録を `/var/backups/kosenmap/retire-ubuntu-日時.txt` に書く(グループ・シェル・期限・AllowUsers)
3. docker・sudo・adm(と lxd などに居ればそれも)から外す
4. 鍵を `authorized_keys.retired-日時` へ退避、パスワードに錠・アカウントを期限切れ・シェルを nologin
5. `AllowUsers` からその名前だけ抜く(`sshd -t` が通らない、または km が消えたら元に戻す)

##### 段 4-1. 検証機で見本を作って、退役 → 戻す → 退役 を通す

検証機には `ubuntu` が居ないので、本番と同じ形(docker・sudo・adm・`AllowUsers ubuntu`)の見本を作ってから流す(`docs/stage4-vm-mimic.sh`。`KM_ENV=local` でなければ止まる)。

🔑 **別の窓で開く** —— sudo のパスワードが要ります。


In [ ]:
%%terminal
scp -O -o IdentitiesOnly=yes -i "$env:USERPROFILE\.ssh\km_ops" "$PWD\..\docs\stage4-vm-mimic.sh" kmops@192.168.217.128:/tmp/stage4-vm-mimic.sh
ssh -t -i "$env:USERPROFILE\.ssh\test" km@192.168.217.128 "sudo sh /tmp/stage4-vm-mimic.sh; read -r -p 'Enter で閉じる' _"


##### 段 4-2. 検証機で確かめる

🟢 **読むだけ** —— 何も変えません。


In [ ]:
%%ps
$vm = '192.168.217.128'
ssh -o BatchMode=yes -i "$env:USERPROFILE\.ssh\test" "km@$vm" "id ubuntu; getent passwd ubuntu | cut -d: -f7; ls -a /home/ubuntu/.ssh; /opt/kosenmap/scripts/host-ops-user.sh status | sed -n '/docker と sudo/,/置き場/p'"
'--- km と kmops は今までどおり'
ssh -o BatchMode=yes -i "$env:USERPROFILE\.ssh\test" "km@$vm" "id -un"
ssh -o BatchMode=yes -o IdentitiesOnly=yes -i "$env:USERPROFILE\.ssh\km_ops" "kmops@$vm" "docker ps -q | wc -l"


**検証機の結果(2026-09-18):** 見本の `ubuntu`(docker・sudo・adm・`AllowUsers ubuntu`)で 退役 → 戻す → 退役 を通した。

| | 退役 | 戻す | 2 回目の退役 |
|---|---|---|---|
| グループ | docker・sudo・adm から外れた | 3 つとも戻った | また外れた(`groups=ubuntu`) |
| 鍵 | `authorized_keys.retired-日時` へ退避 | 戻った | また退避 |
| ログイン | 錠・期限切れ・`nologin` | `/bin/bash` に戻った | `nologin` |
| `AllowUsers` | `km kmops` | `ubuntu` が戻った | `km kmops` |

km・kmops・門番は影響なし。本番の形(`docker ubuntu,kmops`・`sudo ubuntu,km`・`adm syslog,ubuntu`・`AllowUsers ubuntu` の 1 行・sudoers の個別行なし)は見本と同じ。

##### 段 4-3. 本番で ubuntu を退役させる

**戻せる:** 出力の最後に出る `unretire --record …` をそのまま流せば戻る(記録は `/var/backups/kosenmap/`)。

🔑 **別の窓で開く** —— sudo のパスワードが要ります。


In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-ops-user.sh retire; read -r -p 'Enter で閉じる' _"


##### 段 4-4. 本番で確かめる

- `ubuntu` がどのグループにも居ない・`AllowUsers` に居ない・シェルが `nologin`
- km・kmops・門番・配備に影響が無い(配備は後片付けの自己検査まで通す)

🟢 **読むだけ** —— 何も変えません(配備はファイルを置くだけ)。


In [ ]:
%%ps
ssh -o BatchMode=yes -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "id ubuntu; getent passwd ubuntu | cut -d: -f7; /opt/kosenmap/scripts/host-ops-user.sh status"
'--- kmops・門番'
ssh -o BatchMode=yes -o IdentitiesOnly=yes -i "$env:USERPROFILE\.ssh\km_ops" kmops@ito4.jp "docker ps -q | wc -l"
ssh -o BatchMode=yes -o IdentitiesOnly=yes -i "$env:USERPROFILE\.ssh\km_backup" kmops@ito4.jp ping
'--- ubuntu では入れない(Permission denied が正しい)'
ssh -o BatchMode=yes -o ConnectTimeout=10 ubuntu@ito4.jp true 2>&1 | Select-Object -Last 1
$global:LASTEXITCODE = 0


#### B の結果(2026-09-18 本番)

`ubuntu` は `groups=ubuntu,cdrom,dip,plugdev`・`nologin`・`AllowUsers` から外れ、`ubuntu@ito4.jp` は `Permission denied (publickey)`。km・kmops・門番は影響なし。

**いまの形:**

| 利用者 | 鍵(この PC) | docker | sudo | 入れる | 使いみち |
|---|---|---|---|---|---|
| `km` | `km_vps`(パスフレーズ付き・agent) | **無し** | パスワード必須 | 鍵 | 人が入って `sudo host-*.sh` |
| `kmops` | `km_ops`(パスフレーズ付き・agent・`from=60.112.5.32`・転送なし) | あり | **無し** | 鍵 | 配備・`%%host`・`backup-keys.ps1` |
| `kmops`(門番) | `km_backup`(パスフレーズ無し・`restrict,command=ssh-backup-gate.sh`) | ― | ― | 鍵(控えの口だけ) | 週次の控え |
| `ubuntu` | ― | 無し | 無し | **入れない**(退役) | 使わない |
| root | ― | ― | ― | 入れない(`PermitRootLogin no`) | cron の定期の仕事 |

**残る経路:** kmops の鍵(`km_ops`)が盗まれ、**かつパスフレーズが破られ、かつ家の回線から使われる**と、docker で root に届く。
**戻し方:** 段ごとの表の「戻し方」。退役の記録と持ち主の控えは本番の `/var/backups/kosenmap/`(root だけが読める)。

**検証機だけに残っていること:** km が `lxd` に居る(本番は居ない)。本番と揃えるなら `sudo gpasswd -d km lxd`。


## 8. 戻し方の一覧

| 当てたもの | 戻し方 | データへの影響 |
|---|---|---|
| §2 の配備(網・read_only ほか) | 前の版のコードで `deploy-to-host.ps1 -Action up`。§6 を当てていたら**先に** §6 の戻し | 無し(全員が一度ログアウト) |
| §3 Logto の DB 利用者 | `.env` の `LOGTO_DB_USER`・`LOGTO_DB_PASSWORD` を空にして `up -d` | 無し |
| §4 Console のパスワード方針 | SQL で `{}` に戻して Logto を再起動 | 無し |
| §5 別オリジン | `.env` の `KM_ADMIN_DOMAIN` を消して `up -d` | 無し |
| §6 MariaDB | root@localhost で `RENAME USER` を戻す。自分のアカウントは `DROP USER` | 無し |
| §7-1 ACL | `icacls D:\ /restore <控え>` | 無し |
| ランキングの利用者の行(§0) | 控え(`D:\Backups\ito8795.com-20260915-094339`)の MariaDB から `km_map_ranking_users` だけ戻す | 1 行 |